In [1]:
pip install scikit-learn

In [2]:
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

df = pd.read_csv('clean_trading_data.csv')
X = df[['volatility', 'volume_ma']].fillna(0)
y = df['signal'].map({'GREEN':1, 'RED':0})

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

feature_importance = pd.Series(rf.feature_importances_, index=X.columns)
print(feature_importance)


volatility    0.487515
volume_ma     0.512485
dtype: float64


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np

# Load Day 3 enhanced data (uses your features)
df = pd.read_csv('clean_trading_data.csv')

# Day 3 FEATURES (predict GREEN=profitable signals)
feature_cols = ['volatility', 'volume_ma', 'prev_return', 'prev_vol', 
                'price_change', 'volume_spike', 'signal_numeric']

# Fill missing values
X = df[feature_cols].fillna(0)
y = df['signal'].map({'GREEN':1, 'RED':0})  # Binary classification

print(f"Dataset: {X.shape[0]} rows, {len(feature_cols)} features")
print(f"GREEN signals: {y.sum()} ({y.mean():.1%})")

# Train/Test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 100-TREE RANDOM FOREST
rf = RandomForestClassifier(
    n_estimators=100,      # 100 trees
    max_depth=5,           # Prevent overfitting
    min_samples_split=5,   # Stable splits
    random_state=42
)

# TRAIN MODEL
rf.fit(X_train, y_train)

# TEST PERFORMANCE
y_pred = rf.predict(X_test)
print("\n📊 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))

# CROSS-VALIDATION (robust score)
cv_scores = cross_val_score(rf, X, y, cv=5)
print(f"\n🎯 5-FOLD CV ACCURACY: {cv_scores.mean():.3f} (+/- {cv_scores.std()*2:.3f})")

# FEATURE IMPORTANCE (your Day 2 edge shines!)
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\n🏆 FEATURE RANKING:")
print(importance)

# CONFUSION MATRIX
print("\n🔍 CONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred))


KeyError: "['prev_return', 'prev_vol', 'price_change', 'volume_spike', 'signal_numeric'] not in index"

In [4]:
df = pd.read_csv('clean_trading_data.csv')
print("Available columns:", df.columns.tolist())
print("\nDataset shape:", df.shape)
print("\nSample data:")
print(df.head())

Available columns: ['timestamp', 'symbol', 'close', 'volume', 'signal', 'returns', 'volatility', 'volume_ma']

Dataset shape: (90, 8)

Sample data:
          timestamp      symbol    close   volume signal   returns  \
0  18-03-2026 13:35     OLECTRA  1032.45  4953823  GREEN  0.234988   
1  18-03-2026 13:35  CONCORDBIO  1147.90   316437  GREEN  0.111821   
2  18-03-2026 13:35   JINDALSAW   198.92  2388029  GREEN -0.826710   
3  18-03-2026 13:35  ADANIGREEN   896.90  1744215  GREEN  3.508848   
4  18-03-2026 13:35      ENTERO  1134.40   167513  GREEN  0.264801   

   volatility   volume_ma  
0    2.463844  4630097.05  
1    2.168486  4341462.25  
2    2.172398  4437853.30  
3    2.239928  4505350.15  
4    2.203218  4216864.05  


In [10]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

df = pd.read_csv('clean_trading_data.csv')

# Feature engineering (create missing columns)
df['prev_return'] = df['close'].pct_change().shift(1).fillna(0)
df['prev_vol'] = df['volatility'].shift(1).fillna(df['volatility'].mean())
df['price_change'] = df['close'].pct_change().fillna(0)
df['volume_spike'] = (df['volume'] > df['volume'].rolling(20).mean() * 1.5).astype(int)
df['signal_numeric'] = df['signal'].map({'GREEN':1, 'RED':0}).fillna(0)

feature_cols = ['volatility', 'volume_ma', 'prev_return', 'prev_vol', 
                'price_change', 'volume_spike', 'signal_numeric']

X = df[feature_cols].fillna(0)
y = df['signal'].map({'GREEN':1, 'RED':0})

print(f"Dataset: {X.shape[0]} rows, {len(feature_cols)} features")
print(f"GREEN signals: {y.sum()} ({y.mean():.1%})")

# Rest of your code continues unchanged...
# Train/Test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 100-TREE RANDOM FOREST
rf = RandomForestClassifier(
    n_estimators=100,      # 100 trees
    max_depth=5,           # Prevent overfitting
    min_samples_split=5,   # Stable splits
    random_state=42
)

# TRAIN MODEL
rf.fit(X_train, y_train)

# TEST PERFORMANCE
y_pred = rf.predict(X_test)
print("\n📊 CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred))

# CROSS-VALIDATION (robust score)
cv_scores = cross_val_score(rf, X, y, cv=5)
print(f"\n🎯 5-FOLD CV ACCURACY: {cv_scores.mean():.3f} (+/- {cv_scores.std()*2:.3f})")

# FEATURE IMPORTANCE (your Day 2 edge shines!)
importance = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\n🏆 FEATURE RANKING:")
for i, (feature, imp) in enumerate(importance.items(), 1):
    rank_str = f"{i:2d}."
    print(f"{rank_str:4s} {feature:15s} {imp:.6f}")

# CONFUSION MATRIX
print("\n🔍 CONFUSION MATRIX:")
print(confusion_matrix(y_test, y_pred))


Dataset: 90 rows, 7 features
GREEN signals: 45 (50.0%)

📊 CLASSIFICATION REPORT:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        13
           1       1.00      1.00      1.00        14

    accuracy                           1.00        27
   macro avg       1.00      1.00      1.00        27
weighted avg       1.00      1.00      1.00        27


🎯 5-FOLD CV ACCURACY: 1.000 (+/- 0.000)

🏆 FEATURE RANKING:
 1.  signal_numeric  0.688238
 2.  price_change    0.073607
 3.  volume_ma       0.066644
 4.  prev_vol        0.064007
 5.  prev_return     0.057967
 6.  volatility      0.046175
 7.  volume_spike    0.003363

🔍 CONFUSION MATRIX:
[[13  0]
 [ 0 14]]


In [11]:
# PRODUCTION RF: Predict FUTURE profitability (no leakage!)
df['future_return'] = df['returns'].shift(-1)  # Next period return
df['profitable'] = (df['future_return'] > 0).astype(int)  # True target

# PAST FEATURES ONLY (no current signal!)
true_features = ['volatility', 'volume_ma', 'prev_return', 'prev_vol', 
                'price_change', 'volume_spike']  # NO signal_numeric!

X = df[true_features].fillna(0)
y = df['profitable'].dropna()  # Predict future profit
X = X.loc[y.index]

print(f"PRODUCTION MODEL: {len(X)} samples")
print(f"Profitable trades ahead: {y.mean():.1%}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf_production = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_production.fit(X_train, y_train)

y_pred = rf_production.predict(X_test)
print("\n🏭 PRODUCTION PERFORMANCE:")
print(classification_report(y_test, y_pred))
cv_scores = cross_val_score(rf_production, X, y, cv=5)
print(f"CV Accuracy: {cv_scores.mean():.3f}")
print("Features:", pd.Series(rf_production.feature_importances_, index=true_features).sort_values(ascending=False))


PRODUCTION MODEL: 90 samples
Profitable trades ahead: 56.7%

🏭 PRODUCTION PERFORMANCE:
              precision    recall  f1-score   support

           0       0.50      0.42      0.45        12
           1       0.59      0.67      0.62        15

    accuracy                           0.56        27
   macro avg       0.54      0.54      0.54        27
weighted avg       0.55      0.56      0.55        27

CV Accuracy: 0.611
Features: volume_ma       0.221215
price_change    0.202737
prev_return     0.190246
volatility      0.186419
prev_vol        0.162391
volume_spike    0.036991
dtype: float64
